# From forces to a defensible choice

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 18 §18.8 · type: concept-lab*

Turn instinct into explicit trade-off analysis: name a system's **forces**, apply the
**transparency test** as code, read the decision matrix *second*, and record the choice as an
ADR with its exit cost.


## 🧠 Why this matters

“Which framework is best?” is the wrong question; it has no answer. The architect's question is
*which loan of architecture, if any, fits **this** system's forces* — team experience, product
shape, lock-in tolerance, and how far from the happy path you expect to wander. The framework
decision is expensive to reverse (state formats and prompt assembly weave into everything), so
it deserves the same rigor as any architecture decision: name the forces, score against them,
write the ADR. This lab makes that move mechanical so it becomes a habit.


## Objectives & prerequisites

**By the end you can:**

- Name a system's forces *before* naming a framework.
- Apply the transparency test (“could I re-implement this crudely in a week?”) as a function.
- Emit a real ADR stub — forces, decision, rejected options, **exit cost** — for a scenario.

**Prereqs:**

- **18-01** — you've felt the three frameworks in code.
- **Ch 27** — ADRs / trade-off analysis.
- **Ch 28** — hexagonal architecture (keeping your domain out of the framework).

**Offline by design.** This is judgment and structure, not model calls — no API key, ever.


## Setup

There are no model calls here, but we keep the standard switch so the notebook matches every
other one in the course and stays trivially CI-runnable.


In [ ]:
import os
import random
from dataclasses import dataclass, field

from dotenv import load_dotenv

load_dotenv()

MOCK = os.getenv("COMPANION_MOCK", "1") == "1"  # offline either way here
random.seed(18)

print(f"MOCK={MOCK}  (this notebook is offline by design)")


## 1 · Map the landscape

Before scoring anything, get the field straight: what is each option *for*? Fill in your own
one-liner for each before reading ours — if you can't say what a tool is for, you're not ready
to choose it. (These mirror §18.1–§18.6.)


In [ ]:
LANDSCAPE = {
    "LangChain / LangGraph": "graph/state-machine orchestration; checkpointers + interrupts",
    "Pydantic AI / Instructor": "type-safety first; validated model boundary, thin runtime",
    "LlamaIndex": "RAG-centric data plane: ingest, index, retrieve, query over private data",
    "CrewAI / AutoGen": "role/conversation multi-agent; fastest demo, framework owns coordination",
    "Smolagents / DSPy": "code-as-action; prompts-as-compiled-programs (optimization)",
    "Vendor agent SDKs": "best-tuned harness for one vendor's models; least portable",
}

for name, purpose in LANDSCAPE.items():
    print(f"{name:<26} {purpose}")


## 2 · The transparency test, as a function

The one test that separates a safe loan from a trap (§18.1): *could you re-implement what it
does for you, crudely, in a week? Can you read its assembled prompts and wire calls?* Encode it
so “it has a nice landing page” can never be the deciding input.


In [ ]:
@dataclass
class Transparency:
    reimplementable_in_a_week: bool  # could a senior rebuild the core crudely in ~5 days?
    prompts_readable: bool           # can you see the assembled prompt it sends?
    wire_calls_readable: bool        # can you see each provider/tool call it makes?

    def score(self) -> int:
        """0–3. Below 2 means you've likely made core logic un-debuggable."""
        return sum(
            (self.reimplementable_in_a_week, self.prompts_readable, self.wire_calls_readable)
        )

    def verdict(self) -> str:
        s = self.score()
        if s == 3:
            return "leverage — you can see through it"
        if s == 2:
            return "acceptable — know where the dark corner is"
        return "trap — you can't explain what it does on your behalf"


raw_loop = Transparency(True, True, True)
heavy_magic = Transparency(False, False, True)
for name, t in {"raw tool loop": raw_loop, "opaque high-magic": heavy_magic}.items():
    print(f"{name:<20} score={t.score()}/3  -> {t.verdict()}")


## 🔮 Predict the sound default

For each scenario, **predict** the sound default *before* running the next cell. Name the
dominant force first, then guess:

1. A single agent inside a **typed API service**; the team values explicitness.
2. A **long, stateful, multi-step** workflow that must pause for human approval (HIL gates).
3. **RAG over private corpora**; agents are secondary.

Then check against §18.8's matrix below — it encodes the same defaults.


In [ ]:
# The left column — naming forces — is the durable part; the right column changes yearly.
DECISION_MATRIX = {
    "typed API service; team values explicitness":
        "Pydantic AI — or raw SDK + Instructor",
    "long-running, stateful, multi-step; pause/resume; HIL gates":
        "LangGraph with checkpointers",
    "product is RAG over private corpora; agents secondary":
        "LlamaIndex data plane; thin or no agent framework",
    "rapid multi-agent prototyping; content/back-office pipelines":
        "CrewAI — plan the production rewrite up front",
    "all-in on one model vendor; want best-tuned harness fast":
        "that vendor's agent SDK; keep schemas + MCP portable",
    "tiny core loop, few tools, senior team, unusual requirements":
        "no framework — Ch 12–17 are the implementation guide",
    "research on coordination/conversation patterns":
        "AutoGen; Smolagents for code-as-action experiments",
}

for forces, default in DECISION_MATRIX.items():
    print(f"• {forces}\n    → {default}\n")


**What you just saw.** The three predict scenarios map to the first three rows: typed service
→ **Pydantic AI**, long stateful + HIL → **LangGraph + checkpointers**, RAG-first →
**LlamaIndex**. Notice the matrix is keyed on *forces*, not on framework features — you walk in
with a force and walk out with a default, never the reverse.


## ⚠️ Pitfall — choosing by demo / landing page

The most common framework mistake is weighing a framework by its hello-world, not its **worst
day**. The decision input that actually predicts pain is: *what does debugging this look like at
2 a.m.?* The helper below refuses to return a recommendation from demo-shine alone — it forces
the worst-day signals (issue tracker, one request traced through source) into the score.


In [ ]:
def weigh_by_worst_day(
    demo_shine: int,          # 0–5: how good the landing page / hello-world feels (a TRAP input)
    transparency: Transparency,
    issue_tracker_healthy: bool,   # did you read open issues for your use case?
    traced_one_request: bool,      # did you trace a single request through its source?
) -> str:
    if not (issue_tracker_healthy and traced_one_request):
        return "INSUFFICIENT EVIDENCE — you've only seen the demo. Do the worst-day homework first."
    worst_day = transparency.score() + int(issue_tracker_healthy) + int(traced_one_request)
    # demo_shine is deliberately NOT added — it is the bias we're guarding against.
    return f"worst-day score = {worst_day}/5 (demo_shine={demo_shine} ignored by design)"


print(weigh_by_worst_day(5, heavy_magic, issue_tracker_healthy=False, traced_one_request=False))
print(weigh_by_worst_day(2, raw_loop, issue_tracker_healthy=True, traced_one_request=True))


## 3 · The lock-in hedge — invest in the portable artifacts

Whatever you choose, some artifacts travel across **every** framework and vendor SDK here, and
some don't. The portable ones are where to put your effort, because they survive a migration.


In [ ]:
PORTABLE = {
    "tool / handoff schemas (Ch 15/17)": "travel across frameworks and vendor SDKs",
    "MCP tool layer (Ch 19)": "standard every framework here now speaks",
    "prompts, evals, domain logic": "yours if you keep them framework-free (Ch 28)",
}
FRAMEWORK_BOUND = {
    "graph/state definitions": "LangGraph-shaped; rewritten on switch",
    "checkpoint state format": "framework-specific persistence",
    "framework prompt assembly": "hidden, version-coupled",
}

print("Invest here (portable):")
for k, v in PORTABLE.items():
    print(f"  + {k:<38} {v}")
print("\nExpect to re-do on a switch (framework-bound):")
for k, v in FRAMEWORK_BOUND.items():
    print(f"  - {k:<38} {v}")


## 4 · Generate the ADR stub

The architect's deliverable isn't a tweet-length opinion — it's an ADR (Ch 27) that names the
forces, the decision, the **rejected** options, and the **exit cost**. Generate one for a
chosen scenario so the choice is defensible six months later when someone asks “why this?”


In [ ]:
@dataclass
class ADR:
    title: str
    context: str
    forces: list[str]
    decision: str
    rejected: dict[str, str]   # option -> why not
    exit_cost: str
    transparency: Transparency = field(default_factory=lambda: Transparency(True, True, True))

    def render(self) -> str:
        # Build line-by-line so the rendered Markdown is flush-left (no ragged indent).
        lines = [
            f"# ADR: {self.title}",
            "",
            "## Status",
            "Proposed",
            "",
            "## Context",
            self.context,
            "",
            "## Forces",
            *[f"- {f}" for f in self.forces],
            "",
            "## Decision",
            self.decision,
            f"(transparency {self.transparency.score()}/3 — {self.transparency.verdict()})",
            "",
            "## Rejected options",
            *[f"- **{k}** — {v}" for k, v in self.rejected.items()],
            "",
            "## Exit cost",
            self.exit_cost,
        ]
        return "\n".join(lines)


adr = ADR(
    title="Orchestration for the research-agent service",
    context=(
        "A single research agent (search_docs -> cited answer) sits inside our typed FastAPI "
        "service. The team values explicitness and is comfortable owning a tool loop."
    ),
    forces=[
        "Product shape: one agent as a component inside ordinary typed code.",
        "Team skill: senior; values reading its own prompt assembly.",
        "Lock-in tolerance: low — prefer portable schemas + MCP.",
        "Expected weirdness: moderate; needs a validated output boundary.",
    ],
    decision="Pydantic AI for the typed boundary; raw SDK kept in tests as the reference.",
    rejected={
        "LangGraph": "durable graph runtime is ceremony this single-step agent doesn't need yet.",
        "CrewAI": "role/crew model hides coordination; wrong for a typed single agent.",
        "Vendor agent SDK": "best-tuned but least portable; violates the low-lock-in force.",
    },
    exit_cost=(
        "Low: domain (schemas, prompts, tools, evals) is framework-free, so a switch is "
        "re-wiring the thin Pydantic AI layer, not a rewrite. Re-evaluate yearly."
    ),
)

print(adr.render())


**What you just saw.** A complete, defensible ADR stub generated from named forces — including
the two things juniors skip: the **rejected** options (with reasons) and the **exit cost**. Drop
this into `templates/adr-template/` and you have the artifact that makes the choice reviewable.


## 🎯 Senior lens — keep the domain framework-free

The one habit that converts “rewrite” into “re-wire”: keep your **domain** — tool schemas,
handoff contracts, prompts, and evals — in framework-free modules the orchestration layer
merely *calls* (hexagonal, Ch 28). Then the framework is a thin, swappable adapter, not the
spine of your product. Three durable rules close the loop: treat the choice as architecture
(write the ADR), weigh frameworks by their worst day (not the demo), and **re-evaluate yearly,
migrate rarely** — the landscape churns; your system shouldn't chase it. And remember “no
framework” is a real answer with a real cost: you'll rebuild checkpointing, streaming, retries,
and tracing yourself — the honest comparison is their engineering debt versus yours.


## Recap

- The question is never “which framework is best” — it's which loan fits *this* system's forces.
- The **transparency test** is a function: re-implementable in a week + readable prompts +
  readable wire calls. Below 2/3 is a trap.
- Read the **matrix second**: walk in with a force, walk out with a default — never the reverse.
- Weigh by the **worst day** (issue tracker, one traced request), not the landing page.
- Invest in the **portable artifacts** (schemas, MCP, framework-free domain); the ADR records
  the forces, the rejected options, and the **exit cost**.


## Exercises

1. **Score a real candidate.** Pick a framework you've used and fill in a `Transparency(...)`
   honestly. Predict its verdict before calling `.verdict()`; if it's a “trap,” name the one
   thing you'd have to learn to move it to “acceptable.”
2. **Add a force.** The team has zero LangGraph experience and a 2-week deadline. Add that force
   to the ADR and predict whether it flips the decision. Does “team skill” outweigh “durable
   state” here?
3. **A second ADR.** Generate an ADR for scenario 2 (long stateful workflow with HIL gates).
   Predict which row of the matrix it lands on and what its exit cost paragraph must mention
   (hint: graph + checkpoint state formats).
4. **Break the worst-day guard.** Call `weigh_by_worst_day` with dazzling `demo_shine=5` but no
   homework done. Predict the return value and explain why ignoring `demo_shine` is the point.


In [ ]:
# Exercise 1 — score a framework you've actually used


In [ ]:
# Exercise 2 — add a 'team skill / deadline' force; does the decision flip?


In [ ]:
# Exercise 3 — generate a second ADR for the long-stateful-workflow scenario


In [ ]:
# Exercise 4 — demo_shine=5 with no homework: predict the guard's response


## Next

- **Previous notebook:** [`18-01-same-agent-three-ways.ipynb`](./18-01-same-agent-three-ways.ipynb)
  — the code that makes these trade-offs concrete.
- **Template:** drop your ADR stub into
  [`../../../templates/adr-template/`](../../../templates/adr-template/); the framework-choice
  defaults inform [`../../../templates/agent-project-starter/`](../../../templates/agent-project-starter/).
- **Capstone:** this decision sets the division of labor for `capstone/agents/` (raw / graph /
  pydantic_ai); the ADR lands beside checkpoint `checkpoints/ch18-three-ways`.
- **Looking ahead:** the portable hedge — **MCP** (Ch 19) — is what lets every framework here
  share one tool layer; LangGraph **interrupts** (Ch 20) are the approval-gate pattern.
